# MyPortfolioManagement - Complete Tutorial

**Comprehensive portfolio management and optimization using Python**

This notebook demonstrates all major functions in the myPortfolioManagement library.

**Compatible with:** Kaggle, Google Colab

**Repository:** https://github.com/ferhat00/QuantitativePortfolioManagement

---

## Table of Contents

1. [Setup & Installation](#setup)
2. [Data Fetching](#section1)
3. [Returns Calculation](#section2)
4. [Utility Functions](#section3)
5. [Portfolio Optimization](#section4)
6. [Performance Metrics](#section5)
7. [Low-Level Risk Metrics](#section6)
8. [Backtesting & Simulation](#section7)
9. [Bootstrapping](#section8)
10. [Clustering & Analysis](#section9)
11. [Visualization](#section10)
12. [Portfolio Selection](#section11)
13. [Complete Workflow](#section12)

---

## <a id="setup"></a>🚀 Setup & Installation

This section will:
1. Clone the GitHub repository
2. Install dependencies (optimized to avoid conflicts)
3. Set up the environment
4. Verify installation

**⏱️ Estimated time:** 3-5 minutes

In [ ]:
import sys
print(sys.version)

In [ ]:
# 1. Clone and navigate
!git clone https://github.com/msh855/QuantitativePortfolioManagement.git
%cd QuantitativePortfolioManagement
!git checkout fc_func_usage_example_update
!git pull origin fc_func_usage_example_update

In [ ]:
%%writefile requirements-core.txt
pandas>=2.2.0
numpy>=2.0.0,<2.3.0
yfinance==0.2.58
quandl
finvizfinance
yahoofinancials
riskfolio-lib>=6.0.0
pyportfolioopt>=1.5.5
quantstats-lumi>=0.3.0
empyrical-reloaded>=0.5.0
ffn>=1.0.0
pyfolio-reloaded>=0.9.0
matplotlib>=3.9.0
seaborn>=0.13.0
plotly>=5.15.0
scikit-learn>=1.6.0
scipy>=1.14.0
arch>=7.0.0
tslearn>=0.7.0
tsmoothie>=1.0.4
timebudget>=0.1.0
feature-engine>=1.9.0
joblib>=1.5.0
tqdm>=4.65.0
parallel-pandas>=0.7.0

In [ ]:
# 3. Install core packages first
!pip install --upgrade pip
!pip install -r requirements-core.txt

In [ ]:
# 4. Install Ray and OpenBB separately
!pip install ray==2.53.0
!pip install openbb==4.5.0

In [ ]:
# 5. Install the package
!pip install -e .

In [ ]:
# Verify installation and import all necessary libraries
import warnings
warnings.filterwarnings('ignore')

print("🔍 Verifying installation...\n")

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✅ Core libraries (pandas, numpy, matplotlib, seaborn)")
except ImportError as e:
    print(f"❌ Core libraries error: {e}")

try:
    import yfinance as yf
    import riskfolio as rp
    import quantstats_lumi as qs
    from pypfopt import EfficientFrontier
    import ffn
    print("✅ Financial libraries (yfinance, riskfolio, quantstats, pypfopt, ffn)")
except ImportError as e:
    print(f"❌ Financial libraries error: {e}")

try:
    from myPortfolioManagement.myData import get_stock_prices
    from myPortfolioManagement.myReturns import calculate_returns
    from myPortfolioManagement.myPortfolioOptimisation import HRP
    from myPortfolioManagement.myPerformanceMetrics import get_main_stats
    print("✅ myPortfolioManagement package")
except ImportError as e:
    print(f"❌ Package import error: {e}")

# Configure pandas display
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("\n" + "="*80)
print("🎉 SETUP COMPLETE - Ready to start!")
print("="*80)

---
## <a id="section1"></a>📊 Section 1: Data Fetching (myData.py)

Fetch historical price data for a diversified portfolio of ETFs.

In [ ]:
from myPortfolioManagement.myData import (
    get_stock_prices,
    get_stock_info,
    get_option_exp_dates
)

# Define a diversified portfolio of ETFs
tickers = [
    'SPY',   # S&P 500
    'QQQ',   # Nasdaq 100
    'IWM',   # Russell 2000 (Small Cap)
    'EFA',   # International Developed
    'EEM',   # Emerging Markets
    'AGG',   # US Bonds
    'TIP',   # TIPS (Inflation Protected)
    'GLD',   # Gold
    'VNQ',   # Real Estate
    'DBC',   # Commodities
]

print("="*80)
print("SECTION 1: DATA FETCHING")
print("="*80)

print("\n1.1 📥 Fetching stock prices...")
print(f"Tickers: {', '.join(tickers)}")

# Fetch historical prices
prices = get_stock_prices(
    yahoo_tickers=tickers,
    start_date='2018-01-01',
    end_date='2024-12-01',
    freq='daily',
    wide_format=True
)

print(f"\n✅ Success!")
print(f"   Price data shape: {prices.shape}")
print(f"   Date range: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
print(f"   Trading days: {len(prices)}")
print("\n📋 First 5 rows:")
display(prices.head())

In [ ]:
print("1.2 ℹ️ Fetching stock information...")
stock_info = get_stock_info(tickers)
print("\n📋 Stock Information:")
display(stock_info[['yahooTicker', 'longName', 'type', 'currency', 'sector']])

In [ ]:
print("1.3 📈 Visualizing normalized prices...")

# Normalize prices to 100
normalized = prices / prices.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 7))
normalized.plot(ax=ax, linewidth=2)
ax.set_title('Asset Performance (Normalized to 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Index Value', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Data fetching complete!")

---
## <a id="section2"></a>📈 Section 2: Returns Calculation (myReturns.py)

Calculate and analyze returns using different methods and frequencies.

In [ ]:
from myPortfolioManagement.myReturns import (
    calculate_returns,
    average_returns,
    convert_returns_freq,
    calculate_portfolio_returns
)

print("="*80)
print("SECTION 2: RETURNS CALCULATION")
print("="*80)

In [ ]:
print("\n2.1 📊 Calculating daily returns...")
returns_daily = calculate_returns(prices, log_returns=False)
print(f"✅ Daily returns shape: {returns_daily.shape}")
print("\n📋 Daily returns (first 5 rows):")
display(returns_daily.head())

In [ ]:
print("2.2 📊 Calculating log returns...")
returns_log = calculate_returns(prices, log_returns=True)
print("\n📋 Log returns statistics:")
display(returns_log.describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
print("2.3 📅 Converting to monthly returns...")
returns_monthly = calculate_returns(prices, convert_to='monthly')
print(f"✅ Monthly returns shape: {returns_monthly.shape}")
print("\n📋 Monthly returns (last 12 months):")
display(returns_monthly.tail(12))

In [ ]:
print("2.4 💰 Calculating expected returns (different methods)...")

# Historical mean
mu_hist = average_returns(returns_daily, method='hist', periods=252)
print("\n📊 Expected Returns (Historical Mean):")
display(mu_hist.sort_values(ascending=False))

# Exponential Moving Average
mu_ema = average_returns(returns_daily, method='ema', span=500, periods=252)
print("\n📊 Expected Returns (EMA):")
display(mu_ema.sort_values(ascending=False))

In [ ]:
print("2.5 📊 Getting benchmark returns (S&P 500)...")

# Helper function to fix benchmark returns (handles quantstats compatibility)
def get_benchmark_returns_fixed(ticker='^GSPC', name='S&P500'):
    ret_data = qs.utils.download_returns(ticker)
    if isinstance(ret_data, pd.DataFrame):
        ret = ret_data.squeeze()
    else:
        ret = ret_data
    ret.name = name
    return pd.DataFrame(ret)

benchmark_sp500 = get_benchmark_returns_fixed('^GSPC', 'S&P500')
print(f"\n✅ S&P 500 benchmark shape: {benchmark_sp500.shape}")
display(benchmark_sp500.tail())

print("\n✅ Returns calculation complete!")

---
## <a id="section3"></a>🔧 Section 3: Utility Functions (myUtils.py)

Demonstrate utility functions for data manipulation and quality checks.

In [ ]:
from myPortfolioManagement.myUtils import (
    data_overview,
    balance_dates,
    check_date_index,
    cap_outliersTS
)

print("="*80)
print("SECTION 3: UTILITY FUNCTIONS")
print("="*80)

In [ ]:
print("\n3.1 🔍 Data overview (checking data quality)...")
# Convert to long format for data_overview
prices_long = prices.reset_index().melt(
    id_vars='Date', 
    var_name='asset', 
    value_name='price'
)

overview = data_overview(
    prices_long,
    my_assets_col_name='asset',
    my_date_col_name='Date',
    price_col_name='price'
)
print("\n📋 Data Overview:")
display(overview)

In [ ]:
print("3.2 ⚖️ Balancing dates between returns and benchmark...")
returns_balanced, benchmark_balanced = balance_dates(returns_daily, benchmark_sp500)
print(f"✅ Balanced returns shape: {returns_balanced.shape}")
print(f"✅ Balanced benchmark shape: {benchmark_balanced.shape}")

In [ ]:
print("3.3 ✂️ Capping outliers in time series...")
try:
    prices_capped = cap_outliersTS(
        returns_daily, 
        capping_method='iqr', 
        fold=3, 
        plot=False
    )
    print(f"✅ Capped prices shape: {prices_capped.shape}")
except Exception as e:
    print(f"⚠️ Outlier capping: {e}")

print("\n✅ Utility functions complete!")

---
## <a id="section4"></a>🎯 Section 4: Portfolio Optimization (myPortfolioOptimisation.py)

Generate optimal portfolio weights using various optimization methods.

In [ ]:
from myPortfolioManagement.myPortfolioOptimisation import (
    HRP,
    equal_weight_portfolio,
    inverse_vol_portfolio,
    port_GMV,
    port_max_sharpe,
    port_target_return,
    port_target_volatility,
    port_CVAR,
    generate_rp_portfolios,
    risk_contributions
)

print("="*80)
print("SECTION 4: PORTFOLIO OPTIMIZATION")
print("="*80)

# Use returns for optimization (fill NaN with 0)
returns_opt = returns_daily.fillna(0)

In [ ]:
print("\n4.1 ⚖️ Equal Weight Portfolio...")
weights_equal = equal_weight_portfolio(returns_opt)
print("\n📊 Equal Weight Portfolio:")
display(weights_equal)

In [ ]:
print("4.2 📉 Inverse Volatility Portfolio...")
weights_inv_vol = inverse_vol_portfolio(returns_opt, weight_max=0.25)
print("\n📊 Inverse Volatility Portfolio:")
display(weights_inv_vol.sort_values('port_inverse_vol', ascending=False))

In [ ]:
print("4.3 🌳 Hierarchical Risk Parity (HRP)...")
weights_hrp = HRP(
    model='HRP',
    returns_training=returns_opt,
    codependence='pearson',
    covariance='ledoit',
    rm='MV',
    linkage='ward',
    weight_max=0.25,
    weight_min=0.02
)
print("\n📊 HRP Portfolio:")
display(weights_hrp.sort_values('port_weight', ascending=False))

In [ ]:
print("4.4 🌲 HERC (Hierarchical Equal Risk Contribution)...")
weights_herc = HRP(
    model='HERC',
    returns_training=returns_opt,
    codependence='pearson',
    rm='CVaR',
    weight_max=0.30,
    weight_min=0.02
)
print("\n📊 HERC Portfolio:")
display(weights_herc.sort_values('port_weight', ascending=False))

In [ ]:
print("4.5 📉 Global Minimum Variance Portfolio...")
weights_gmv = port_GMV(
    returns_training=returns_opt,
    weight_min=0.02,
    weight_max=0.30
)
print("\n📊 Global Minimum Variance Portfolio:")
display(weights_gmv.sort_values('port_min_vol', ascending=False))

In [ ]:
print("4.6 📈 Maximum Sharpe Ratio Portfolio...")
weights_max_sharpe = port_max_sharpe(
    returns_training=returns_opt,
    rf=0.04,
    weight_min=0.02,
    weight_max=0.30
)
print("\n📊 Max Sharpe Portfolio:")
display(weights_max_sharpe.sort_values('port_max_Sharpe', ascending=False))

In [ ]:
print("4.7 🛡️ Minimum CVaR Portfolio...")
weights_cvar = port_CVAR(
    returns_training=returns_opt,
    confidence_interval=0.95,
    rf=0.04,
    weight_min=0.02,
    weight_max=0.30
)
print("\n📊 Minimum CVaR Portfolio:")
display(weights_cvar.sort_values('port_target_CVAR', ascending=False))

In [ ]:
# Compare all portfolios
# First, let's inspect the structure of each weights dataframe
print("Inspecting weight dataframes...")
print("\nweights_equal columns:", weights_equal.columns.tolist())
print("weights_inv_vol columns:", weights_inv_vol.columns.tolist())
print("weights_hrp columns:", weights_hrp.columns.tolist())
print("weights_herc columns:", weights_herc.columns.tolist())
print("weights_gmv columns:", weights_gmv.columns.tolist())
print("weights_max_sharpe columns:", weights_max_sharpe.columns.tolist())
print("weights_cvar columns:", weights_cvar.columns.tolist())

# Now build the comparison dataframe more robustly
all_weights = pd.DataFrame(index=weights_hrp.index)

# Extract weights, handling different column names
all_weights['Equal'] = weights_equal['port_naive'].values if 'port_naive' in weights_equal.columns else weights_equal.iloc[:, -1].values
all_weights['Inv_Vol'] = weights_inv_vol['port_inverse_vol'].values if 'port_inverse_vol' in weights_inv_vol.columns else weights_inv_vol.iloc[:, -1].values
all_weights['HRP'] = weights_hrp['port_weight'].values if 'port_weight' in weights_hrp.columns else weights_hrp.iloc[:, -1].values
all_weights['HERC'] = weights_herc['port_weight'].values if 'port_weight' in weights_herc.columns else weights_herc.iloc[:, -1].values

# For GMV, Max_Sharpe, and CVaR - handle cases where 'asset' might be index or column
if 'asset' in weights_gmv.columns:
    all_weights['GMV'] = weights_gmv.set_index('asset')['port_min_vol']
else:
    all_weights['GMV'] = weights_gmv['port_min_vol'].values if 'port_min_vol' in weights_gmv.columns else weights_gmv.iloc[:, -1].values

if 'asset' in weights_max_sharpe.columns:
    all_weights['Max_Sharpe'] = weights_max_sharpe.set_index('asset')['port_max_Sharpe']
else:
    all_weights['Max_Sharpe'] = weights_max_sharpe['port_max_Sharpe'].values if 'port_max_Sharpe' in weights_max_sharpe.columns else weights_max_sharpe.iloc[:, -1].values

if 'asset' in weights_cvar.columns:
    all_weights['Min_CVaR'] = weights_cvar.set_index('asset')['port_target_CVAR']
else:
    all_weights['Min_CVaR'] = weights_cvar['port_target_CVAR'].values if 'port_target_CVAR' in weights_cvar.columns else weights_cvar.iloc[:, -1].values

print("\nAll Portfolio Weights Comparison:")
all_weights.round(3)

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
all_weights.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Portfolio Weights Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Asset', fontsize=12)
ax.set_ylabel('Weight', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n✅ Portfolio optimization complete!")

---
## <a id="section5"></a>📊 Section 5: Performance Metrics (myPerformanceMetrics.py)

Calculate comprehensive risk and return metrics.

In [ ]:
from myPortfolioManagement.myPerformanceMetrics import (
    get_main_stats,
    alpha_beta_table,
    alpha_beta_bull,
    alpha_beta_bear,
    information_ratio,
    drawdown_details,
    assets_drawdown_details,
    performance_overview,
    cagr,
    beta_Co_Moments_table
)

print("="*80)
print("SECTION 5: PERFORMANCE METRICS")
print("="*80)

In [ ]:
print("\n5.1 📊 Main Performance Statistics...")
main_stats = get_main_stats(returns_daily, rf=0.04, smart=True)
print("\n📋 Main Performance Stats:")
display(main_stats.round(4))

In [ ]:
print("5.2 📈 CAGR Calculation...")
cagr_values = cagr(prices)
print("\n📋 Compound Annual Growth Rate:")
display(cagr_values.sort_values(ascending=False).round(4))

In [ ]:
print("5.3 🎯 Alpha-Beta Analysis (Full Market)...")
ab_table = alpha_beta_table(
    returns=returns_daily,
    returns_benchmark=benchmark_sp500,
    rf=0.04
)
print("\n📋 Alpha-Beta Table:")
display(ab_table)

In [ ]:
print("5.4 📉 Drawdown Details (for SPY)...")
spy_prices = prices.iloc[:, 0]
dd_details = drawdown_details(spy_prices, top_drawdowns=5)
print(f"\n📋 Top 5 Drawdowns for {prices.columns[0]}:")
display(dd_details)

print("\n✅ Performance metrics complete!")

---
## <a id="section6"></a>🔬 Section 6: Low-Level Risk Metrics (metrics.py)

Calculate individual risk metrics using low-level functions.

In [ ]:
from myPortfolioManagement.metrics import (
    vol, beta, var, cvar, lpm, hpm,
    max_dd, dd, sharpe_ratio, sortino_ratio,
    treynor_ratio, calmar_ratio, omega_ratio,
    gain_loss_ratio, upside_potential_ratio
)

print("="*80)
print("SECTION 6: LOW-LEVEL RISK METRICS")
print("="*80)

# Use SPY returns for demonstration
spy_returns = returns_daily.iloc[:, 0].dropna().values
market_returns = benchmark_sp500.squeeze().dropna().values

# Align lengths
min_len = min(len(spy_returns), len(market_returns))
spy_returns = spy_returns[:min_len]
market_returns = market_returns[:min_len]

In [ ]:
print("\n6.1 📊 Basic Risk Metrics...")
print(f"Volatility (annualized): {vol(spy_returns) * np.sqrt(252):.4f}")
print(f"Beta to market: {beta(spy_returns, market_returns):.4f}")

In [ ]:
print("6.2 💰 Value at Risk...")
print(f"VaR (5%): {var(spy_returns, 0.05):.4f}")
print(f"CVaR (5%): {cvar(spy_returns, 0.05):.4f}")

In [ ]:
print("6.3 📉 Drawdown Metrics...")
print(f"Maximum Drawdown: {max_dd(spy_returns):.4f}")
print(f"Drawdown (30 days): {dd(spy_returns, 30):.4f}")

In [ ]:
print("6.4 📈 Risk-Adjusted Return Ratios...")
er = np.mean(spy_returns) * 252  # Annualized expected return
rf = 0.04  # Risk-free rate

print(f"Sharpe Ratio: {sharpe_ratio(er, spy_returns, rf/252):.4f}")
print(f"Sortino Ratio: {sortino_ratio(er, spy_returns, rf/252):.4f}")
print(f"Treynor Ratio: {treynor_ratio(er, spy_returns, market_returns, rf/252):.4f}")
print(f"Calmar Ratio: {calmar_ratio(er, spy_returns, rf/252):.4f}")
print(f"Omega Ratio: {omega_ratio(er, spy_returns, rf/252):.4f}")

print("\n✅ Risk metrics complete!")

---
## <a id="section7"></a>🔄 Section 7: Backtesting & Simulation (myBacktesting.py)

Test portfolio strategies using Monte Carlo and bootstrap methods.

In [ ]:
from myPortfolioManagement.myBacktesting import (
    bootstrap_stats,
    bootstrap_portfolio_performance,
    sim_series,
    beating_probability,
    performance
)

print("="*80)
print("SECTION 7: BACKTESTING & SIMULATION")
print("="*80)

# Create a simple portfolio return series for backtesting
portfolio_returns = returns_daily.mean(axis=1)  # Equal weighted proxy
portfolio_returns.name = 'Portfolio'

In [ ]:
print("\n7.1 🎲 Bootstrap Statistics (500 simulations)...")
print("This may take 30-60 seconds...")

bootstrap_results = bootstrap_stats(
    returns=portfolio_returns,
    returns_benchmark=benchmark_sp500.squeeze(),
    rf=0.04,
    periods=252,
    n_sim=500
)
print("\n📊 Bootstrap Statistics Distribution:")
display(bootstrap_results.describe().round(4))

In [ ]:
print("7.2 📊 Bootstrap Portfolio Performance (In-Sample vs Out-of-Sample)...")
print("This may take 1-2 minutes...")

try:
    means, distributions, dist_stats = bootstrap_portfolio_performance(
        returns=portfolio_returns,
        returns_benchmark=benchmark_sp500.squeeze(),
        periods=252,
        rf=0.04,
        out_of_sample_date='2023-01-01',
        n_sim=500
    )
    print("\n📋 Mean Performance Metrics:")
    display(means.round(4))
except Exception as e:
    print(f"⚠️ Bootstrap performance analysis: {e}")

In [ ]:
print("7.3 🎯 Probability of Beating Benchmark...")
print("Running Monte Carlo simulation...")

try:
    beat_prob = beating_probability(
        returns=pd.DataFrame(portfolio_returns),
        returns_benchmark=benchmark_sp500,
        n_sample=500
    )
    prob_value = beat_prob.values[0][0]
    print(f"\n✅ Probability of beating S&P 500: {prob_value:.1%}")
except Exception as e:
    print(f"⚠️ Beating probability calculation: {e}")

print("\n✅ Backtesting complete!")

---
## <a id="section8"></a>🔁 Section 8: Bootstrapping (myBootstrapping.py)

Resample data using various bootstrap methods.

In [ ]:
from myPortfolioManagement.myBootstrapping import (
    BootstrapIDD,
    BootstrapStationary,
    BootstrapCircular,
    BootstrapMovingBlock,
    bootstrappingTS
)

print("="*80)
print("SECTION 8: BOOTSTRAPPING")
print("="*80)

# Use single asset returns for bootstrapping
single_returns = returns_daily.iloc[:, 0].dropna()

In [ ]:
print("\n8.1 🎲 IID Bootstrap...")
bs_iid = BootstrapIDD(series=single_returns, n_samples=100, seed=42)
print(f"✅ IID Bootstrap shape: {bs_iid.shape}")
print(f"Original std: {single_returns.std():.6f}")
print(f"Bootstrap mean std: {bs_iid.std().mean():.6f}")

In [ ]:
print("8.2 📊 Stationary Bootstrap...")
bs_stationary = BootstrapStationary(
    series=single_returns,
    block_size=20,
    n_samples=100,
    seed=42
)
print(f"✅ Stationary Bootstrap shape: {bs_stationary.shape}")

print("\n✅ Bootstrapping complete!")

---
## <a id="section9"></a>🔍 Section 9: Clustering & Analysis (myClustering.py)

Group similar assets and detect market regimes.

In [ ]:
from myPortfolioManagement.myClustering import (
    ts_clustering,
    cluster_ftca,
    detect_regimes
)

print("="*80)
print("SECTION 9: CLUSTERING & ANALYSIS")
print("="*80)

In [ ]:
print("\n9.1 🔗 Time Series Clustering (DTW)...")

try:
    clusters, cluster_centers = ts_clustering(
        df=prices,
        number_of_clusters=3,
        algo='dtw',
        plot_bar_center=False
    )
    print("\n📊 Asset Clusters:")
    display(clusters)
    print(f"\n✅ Cluster Centers shape: {cluster_centers.shape}")
except Exception as e:
    print(f"⚠️ Time series clustering: {e}")

In [ ]:
print("9.2 ⚡ Fast Threshold Clustering Algorithm (FTCA)...")

try:
    ftca_clusters = cluster_ftca(
        returns=returns_daily,
        threshold=0.50,
        col_name='asset'
    )
    print("\n📊 FTCA Clusters:")
    display(ftca_clusters)
except Exception as e:
    print(f"⚠️ FTCA clustering: {e}")

print("\n✅ Clustering complete!")

---
## <a id="section10"></a>📊 Section 10: Visualization (myPlots.py)

Create comprehensive charts and plots.

In [ ]:
from myPortfolioManagement.myPlots import (
    correlation_matrix,
    monthly_heatmap,
    scatter_plot_simple
)

print("="*80)
print("SECTION 10: VISUALIZATION")
print("="*80)

In [ ]:
print("\n10.1 🔗 Correlation Matrix...")
correlation_matrix(
    returns_daily,
    corr_limit=None,
    figsize=(12, 8),
    diagonal=True
)
plt.show()

In [ ]:
print("10.2 📅 Monthly Returns Heatmap...")
try:
    fig = monthly_heatmap(
        returns=portfolio_returns,
        annot_size=8,
        figsize=(12, 8),
        cbar=True,
        eoy=True,
        show=True
    )
except Exception as e:
    print(f"⚠️ Monthly heatmap: {e}")

In [ ]:
# Risk-Return Scatter Plot
risk_return = pd.DataFrame({
    'Return': returns_daily.mean() * 252,
    'Volatility': returns_daily.std() * np.sqrt(252)
})

scatter_plot_simple(risk_return, x='Volatility', y='Return')
plt.title('Risk-Return Profile')
plt.show()

In [ ]:
print("10.3 📈 Portfolio Cumulative Returns Comparison...")

# Calculate returns for different strategies
portfolio_comparison = pd.DataFrame({
    'Equal_Weight': (returns_daily * pd.Series(weights_equal['port_naive'].to_dict())).sum(axis=1),
    'HRP': (returns_daily * pd.Series(weights_hrp['port_weight'].to_dict())).sum(axis=1),
    'Max_Sharpe': (returns_daily * pd.Series(weights_max_sharpe['port_max_Sharpe'].to_dict())).sum(axis=1),  # Fixed: removed .set_index('asset')
    'SPY_Benchmark': returns_daily.iloc[:, 0]
})

# Cumulative returns
cum_returns = (1 + portfolio_comparison).cumprod()

fig, ax = plt.subplots(figsize=(14, 7))
cum_returns.plot(ax=ax, linewidth=2)
ax.set_title('Cumulative Returns: Portfolio Strategies Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Growth of $1', fontsize=12)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📋 Final Values (Growth of $1):")
display(cum_returns.iloc[-1].round(4))
print("\n✅ Visualization complete!")

---
## <a id="section11"></a>🎲 Section 11: Portfolio Selection (myPortfolioSelection.py)

Generate all possible portfolio combinations.

In [ ]:
from myPortfolioManagement.myPortfolioSelection import (
    possible_combinations,
    all_combinations_inner_loop
)

print("="*80)
print("SECTION 11: PORTFOLIO SELECTION")
print("="*80)

In [ ]:
print("\n11.1 🎲 Generating all possible portfolio combinations...")
assets = ['SPY', 'QQQ', 'IWM', 'AGG', 'GLD']

# All combinations of 2-4 assets
combinations = possible_combinations(
    assets_to_consider=assets,
    min_assets=2,
    max_assets=4
)
print(f"\n✅ Number of possible portfolios (2-4 assets): {len(combinations)}")
print("\nFirst 10 combinations:")
for i, combo in enumerate(combinations[:10]):
    print(f"  {i+1}. {combo}")

In [ ]:
print("11.2 🎯 Combinations with must-have assets...")
combinations_must_have = possible_combinations(
    assets_to_consider=assets,
    min_assets=3,
    max_assets=4,
    must_have=['SPY', 'AGG']  # Must include these
)
print(f"\n✅ Portfolios that must include SPY and AGG: {len(combinations_must_have)}")
for combo in combinations_must_have:
    print(f"  {combo}")

print("\n✅ Portfolio selection complete!")

---
## <a id="section12"></a>🎯 Section 12: Complete Workflow - End to End Example

This section demonstrates a complete portfolio management workflow:
1. Fetch data
2. Calculate returns
3. Optimize portfolio
4. Backtest strategy
5. Analyze performance

In [ ]:
print("="*80)
print("SECTION 12: COMPLETE WORKFLOW - END TO END EXAMPLE")
print("="*80)

# Step 1: Data already fetched above
print("\n✅ Step 1: Using previously fetched data...")
print(f"  Assets: {list(prices.columns)}")
print(f"  Period: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")

In [ ]:
# Step 2: Split into training and testing
split_date = '2023-01-01'
returns_train = returns_daily[returns_daily.index < split_date]
returns_test = returns_daily[returns_daily.index >= split_date]
print(f"\n✅ Step 2: Train/Test Split at {split_date}")
print(f"  Training: {len(returns_train)} days")
print(f"  Testing: {len(returns_test)} days")

In [ ]:
# Step 3: Optimize portfolios on training data
print("\n✅ Step 3: Optimizing portfolios on training data...")
weights_hrp_train = HRP(
    model='HRP',
    returns_training=returns_train.fillna(0),
    weight_max=0.25,
    weight_min=0.02
)
print("  HRP weights optimized")

weights_equal_train = equal_weight_portfolio(returns_train)
print("  Equal weight portfolio created")

In [ ]:
# Step 4: Calculate out-of-sample returns
print("\n✅ Step 4: Calculating out-of-sample returns...")

# HRP portfolio returns
hrp_weights_dict = weights_hrp_train['port_weight'].to_dict()
portfolio_hrp_test = (returns_test * pd.Series(hrp_weights_dict)).sum(axis=1)
portfolio_hrp_test.name = 'HRP'

# Equal weight returns
equal_weights_dict = weights_equal_train['port_naive'].to_dict()
portfolio_equal_test = (returns_test * pd.Series(equal_weights_dict)).sum(axis=1)
portfolio_equal_test.name = 'Equal_Weight'

# Combine for comparison
portfolio_comparison = pd.DataFrame({
    'HRP': portfolio_hrp_test,
    'Equal_Weight': portfolio_equal_test,
    'SPY_Benchmark': returns_test.iloc[:, 0]  # SPY as benchmark
})

In [ ]:
# Step 5: Performance analysis
print("\n✅ Step 5: Out-of-Sample Performance Analysis...")
test_stats = get_main_stats(portfolio_comparison, rf=0.04)
print("\n📋 Out-of-Sample Performance Metrics:")
display(test_stats.round(4))

# Cumulative returns
cum_returns = (1 + portfolio_comparison).cumprod()
print("\n📋 Cumulative Returns (End of Period):")
display(cum_returns.iloc[-1].round(4))

---
## 📊 Final Summary

Comprehensive analysis summary and key findings.

In [ ]:
print("\n" + "="*80)
print("SUMMARY & KEY FINDINGS")
print("="*80)

print(f"""
🎉 Portfolio Management Analysis Complete!

📊 DATA ANALYZED:
- Assets: {len(tickers)} ETFs across multiple asset classes
- Period: {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}
- Trading Days: {len(prices)}

🎯 OPTIMIZATION STRATEGIES TESTED:
- Equal Weight (Naive 1/N)
- Inverse Volatility
- Hierarchical Risk Parity (HRP)
- Hierarchical Equal Risk Contribution (HERC)
- Global Minimum Variance (GMV)
- Maximum Sharpe Ratio
- Minimum CVaR

📈 BEST PERFORMING ASSET:
- Ticker: {cagr_values.idxmax()}
- CAGR: {cagr_values.max():.2%}

📉 WORST PERFORMING ASSET:
- Ticker: {cagr_values.idxmin()}
- CAGR: {cagr_values.min():.2%}

💡 NEXT STEPS:
1. Review portfolio weights and adjust constraints
2. Run more extensive backtests with different parameters
3. Implement rebalancing strategy
4. Add transaction costs and slippage
5. Consider regime-based allocation

✅ All sections completed successfully!
""")

print("="*80)
print("END OF TUTORIAL")
print("="*80)

---
## 📚 Additional Resources

**Documentation:**
- [GitHub Repository](https://github.com/ferhat00/QuantitativePortfolioManagement)
- [Riskfolio-Lib Docs](https://riskfolio-lib.readthedocs.io/)
- [PyPortfolioOpt Docs](https://pyportfolioopt.readthedocs.io/)

**Further Reading:**
- Hierarchical Risk Parity (López de Prado, 2016)
- Modern Portfolio Theory (Markowitz, 1952)
- Risk-based and Factor Investing (Ang, 2014)

**Key Concepts Demonstrated:**
1. ✅ Data fetching and preprocessing
2. ✅ Returns calculation (simple, log, different frequencies)
3. ✅ Portfolio optimization (7+ methods)
4. ✅ Performance metrics and risk analysis
5. ✅ Backtesting and Monte Carlo simulation
6. ✅ Asset clustering and regime detection
7. ✅ Comprehensive visualization
8. ✅ Complete end-to-end workflow

---
*Created with ❤️ for quantitative portfolio management*

*Compatible with Kaggle and Google Colab*